In [1]:
import tiktoken
import torch

In [2]:
batch = 2
no_tokens=6
dim = 4

batch = torch.rand(batch, no_tokens, dim)

In [21]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, in_dim, out_dim, heads, context_len, dropout):
        super().__init__()
        assert(out_dim % heads == 0.)
        self.out_dim = out_dim
        self.heads = heads
        self.h_dim = out_dim//heads 
        
        self.Wq = nn.Linear(in_dim, out_dim)
        self.Wk = nn.Linear(in_dim, out_dim)
        self.Wv = nn.Linear(in_dim, out_dim)

        self.out_proj = nn.Linear(out_dim, out_dim)

        self.dropout = nn.Dropout(dropout)

        self.register_buffer("mask", torch.triu(torch.ones(context_len, context_len), diagonal=1))
        
    def forward(self, x):
        B, T, D = x.shape 

        query = self.Wq(x).reshape(B, T, self.heads, self.h_dim) 
        keys = self.Wk(x).reshape(B, T, self.heads, self.h_dim)
        values = self.Wv(x).reshape(B, T, self.heads, self.h_dim)

        query = query.transpose(1,2) # B, H, T, D
        keys = keys.transpose(1,2)
        values = values.transpose(1,2)

        attn_scores = query @ keys.transpose(-1, -2)
        mask_bool = self.mask.bool()[:T, :T]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores/keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vect = (attn_weights @ values).transpose(1,2).reshape(B, T, D)

        context_vect = self.out_proj(context_vect)

        return context_vect
        


In [4]:
in_dim= 4
out_dim = 4
context_len = 6 
B, T, D = 2, 6, 4

Bs = torch.rand(B,T,D)

mha = MultiHeadAttention(in_dim, out_dim, 2, context_len, 0.5)
mha(Bs)

tensor([[[-0.3899, -0.4681, -0.4499, -0.2346],
         [-0.6122, -0.6697,  0.1624,  0.0829],
         [-0.5546, -0.5773, -0.1041,  0.0184],
         [-0.5714, -0.6985,  0.2718,  0.0041],
         [-0.5585, -0.6262,  0.0347,  0.0010],
         [-0.4495, -0.6115, -0.1206, -0.1581]],

        [[-0.6951, -0.6691,  0.9304,  0.1907],
         [-0.6026, -0.5692,  0.6067,  0.0515],
         [-0.4657, -0.4838,  0.0265, -0.1312],
         [-0.5829, -0.6640,  0.5787,  0.0321],
         [-0.6282, -0.8841,  0.7606,  0.0924],
         [-0.4895, -0.6485,  0.0893, -0.0917]]], grad_fn=<ViewBackward0>)

In [5]:
class GPTConfig: 
    def __init__(
        self,
        vocab_size= 50257,   
        context_length= 1024,
        emb_dim= 768,       
        n_heads= 12,       
        n_layers =12,     
        drop_rate= 0.1,  
        qkv_bias= False
    ):
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.emb_dim = emb_dim
        self.n_heads = n_heads 
        self.n_layers = n_layers
        self.drop_rate = drop_rate 
        self.qkv_bias = qkv_bias 


In [6]:
import torch.nn as nn

class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        config = config()
        
        self.tok_emb = nn.Embeddings(config.vocab_size, config.emb_dim)
        self.pos_emb = nn.Embeddings(config.vocab_size, config.emb_dim)
        self.drop_emb = nn.Embeddings(config.vocab_size, config.emb_dim)

        self.trf_block = nn.Sequential(*[TransformerBlock(config) for i in range(config.n_layers)])

        self.final_norm = LayerNorm(config)

        self.out_head = nn.Linear(config.emb_dim, config.vocab_size, bias=False)


    def forward(self, x):
        B, T, D = x.shape

        tok_embeds = self.tok_emb(x)
        pos_embeds = self.pos_emb(x)

        x = tok_embeds + pos_embeds 
        x = self.drop_emb(x)
        x = self.trf_block(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits 

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        config = config()

        self.mha = MultiHeadAttention(config.in_dim, config.out_dim, config.n_heads, config.context_len, config.dropout)
        self.l_norm =  LayerNorm()
        self.ff = FeedForward()

    def forward(self, x):
        x = self.mha(x)
        x = self.l_norm(x)
        x = self.ff(x)

        return x

In [8]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
                                  
    def forward(self, x):
        mean = torch.mean(x, dim=-1, keepdim=True)
        var = torch.var(x, dim=-1, keepdim=True, correction=0)
        norm_x = (x - mean)/torch.sqrt(var+self.eps)
        return (norm_x * self.scale) + self.shift 

    

In [9]:
class GELU(nn.Module):
    def __init__(self ):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))
        
        
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        cfg = config
        self.layers = nn.Sequential(nn.Linear(cfg.emb_dim, 4 * cfg.emb_dim), GELU(), nn.Linear(4 * cfg.emb_dim, cfg.emb_dim))

    def forward(self, x):
        return self.layers(x)
         

In [10]:
config = GPTConfig()
ff = FeedForward(config)
x = torch.rand(2, 3, 768)
out = ff(x)
out.shape

torch.Size([2, 3, 768])

In [11]:
config = {
    "emb_dim": 768, # double check this
    "vocab_size": 50127,
    "context_len": 1024,
    "n_heads": 12,
    "n_layers":12,
    "dropout_rate": 0.1,
    "qkv_bias":False,
}


In [12]:
r = torch.rand(3,3)
l = LayerNorm(3)
l(r)


tensor([[ 0.5477, -1.4024,  0.8547],
        [ 0.3159,  1.0356, -1.3515],
        [ 0.6761, -1.4137,  0.7376]], grad_fn=<AddBackward0>)

In [22]:
import torch.nn as nn 

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))
        
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"]))
        
    def forward(self, x):
        return self.layers(x)



class LayerNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.eps = 1e-5
        self.shift = nn.Parameter(torch.zeros(dim))
        self.scale = nn.Parameter(torch.ones(dim))
        
    def forward(self, x):
        mean = torch.mean(x, dim=-1, keepdim=True)
        var = torch.var(x, dim=-1, keepdim=True, correction=0)

        norm_x = (x-mean/var+self.eps)
        return (x * self.scale) + self.shift 


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mha = MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["n_heads"], cfg["context_len"], cfg["dropout_rate"]
                                     )
        self.ffn = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["dropout_rate"])

    def forward(self, x):
        shortcut = x 
        x = self.norm1(x)
        x = self.mha(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = self.drop_shortcut(x)
        x = x+shortcut 
        
        return x

    
# class MultiHeadAttention(nn.Module):
#     def __init__(self, config):
#         super().__init__()
#         d_in = config["emb_dim"]
#         d_out = config["emb_dim"]
#         context_len = config["context_len"]
#         bias = config["qkv_bias"]
        
#         self.heads = config["n_heads"]
#         self.h_dim = d_out//self.heads 
        
#         self.Wq = nn.Linear(d_in, d_out, bias)
#         self.Wk = nn.Linear(d_in, d_out, bias)
#         self.Wv = nn.Linear(d_in, d_out, bias)
#         self.dropout = nn.Dropout(config["dropout_rate"])
        
#         self.register_buffer("mask", torch.triu(torch.ones(context_len, context_len), diagonal=1))
                             
#     def forward(self, x):
#         B, T, D = x.shape

#         query = self.Wq(x).reshape(B, T, self.heads, self.h_dim) 
#         keys = self.Wk(x).reshape(B, T, self.heads, self.h_dim) 
#         values = self.Wv(x).reshape(B, T, self.heads, self.h_dim) 

#         query = query.transpose(1,2)
#         keys = keys.transpose(1,2)
#         values = values.transpose(1,2)

#         att_scores = query @ keys.transpose(-1,-2)

#         att_scores = self.dropout(att_scores)

#         mask_bool = self.mask.bool()

#         att_scores.masked_fill_(mask_bool, -torch.inf)

#         attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
#         attn_weights = self.dropout(attn_weights)

#         context_vec = (attn_weights @ values).transpose(1,2)

#         context_vec = context_vec.reshape(B, T, D)

#         return context_vece

In [14]:
ffn = FeedForward(config)

x = torch.rand(2,3,768)
out = ffn(x)
out.shape


torch.Size([2, 3, 768])

In [1]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_enc = nn.Embedding(cfg["context_len"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["dropout_rate"]) 

        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_enc(torch.arange(seq_len, device=in_idx.device)) #unsq here at 0
        
        x  = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x  = self.final_norm(x)
        logits = self.out_head(x)
        return logits 

NameError: name 'nn' is not defined

In [16]:
batch

tensor([[[0.9905, 0.5348, 0.4501, 0.8923],
         [0.4160, 0.6215, 0.9754, 0.0992],
         [0.6899, 0.8314, 0.4303, 0.8636],
         [0.3651, 0.7762, 0.2652, 0.2283],
         [0.2222, 0.7777, 0.7639, 0.9054],
         [0.1775, 0.4732, 0.6223, 0.7875]],

        [[0.5902, 0.9853, 0.5927, 0.4318],
         [0.6006, 0.3796, 0.6406, 0.7888],
         [0.4787, 0.1369, 0.6520, 0.6721],
         [0.1014, 0.0412, 0.0317, 0.9752],
         [0.5827, 0.9901, 0.0322, 0.4696],
         [0.8990, 0.0424, 0.1546, 0.2300]]])

In [24]:
torch.manual_seed(123)
model = GPTModel(config)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[1417, 1547, 1830,  275],
        [ 388,  697,  101, 1935]])

Output shape: torch.Size([2, 4, 50127])
tensor([[[-2.1332,  2.0536,  2.6373,  ..., -1.3888,  0.3572,  4.2805],
         [-5.7865,  1.0264,  2.4641,  ...,  0.1091,  1.1763,  5.7230],
         [-3.3467, -0.1519,  3.8150,  ..., -0.9110, -0.9773, -0.5911],
         [-1.4885,  2.1248,  1.3179,  ..., -1.4113,  1.3423,  0.8532]],

        [[-3.6130, -2.3953,  2.3831,  ..., -2.0477,  3.2899,  7.2384],
         [-0.8221, -2.5189,  4.3264,  ...,  4.2769,  1.3526,  3.8471],
         [-1.7281,  0.0451,  2.8579,  ...,  2.0502, -1.5478,  0.6788],
         [-1.7046,  0.9534,  3.9924,  ...,  1.1456, -1.9286,  4.3236]]],
       grad_fn=<UnsafeViewBackward0>)


In [28]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [31]:
import tiktoken

tokenizer = tiktoken.encoding_for_model("gpt2")
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


In [1]:
model.eval()

out = generate_text_sample(
    model=model,
    idx = encoded_tensor,
    max_new_tokens = 6,
    context_size=GPT_CONFIG_124M["context_len"]
)


NameError: name 'model' is not defined